# Phase 0 — API-Ready Four-Function Architecture

This notebook is intentionally organized for the next implementation step,
where each main function can become an independent API endpoint.

## Main services

1. `extract_objects_from_ebook(chapter_id, detector)`
2. `classify_image(image_url, classifier)`
3. `process_hotspot_image(image_url, hotspot_model, image_ocr_text="")`
4. `generate_toc(extraction_result)`

The model instances are loaded once outside the functions so that a future
API server can reuse them across requests.

The current Colab orchestration routes an image to `process_hotspot_image()`
only when `classify_image()` identifies it as `hotspot_image`.


## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Clone YOLOv5 only once per fresh Colab runtime
!git clone https://github.com/ultralytics/yolov5.git

%cd /content/yolov5

!pip install -q -r requirements.txt
!pip install -q easyocr tqdm pillow requests ultralytics opencv-python tensorflow

Cloning into 'yolov5'...
remote: Enumerating objects: 18752, done.
remote: Counting objects: 100% (312/312), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 18752 (delta 255), reused 130 (delta 130), pack-reused 18440 (from 4)
Receiving objects: 100% (18752/18752), 17.77 MiB | 20.77 MiB/s, done.
Resolving deltas: 100% (12758/12758), done.
/content/yolov5


## 2. Imports

In [ ]:
import json
import re
import shutil
import traceback
from copy import deepcopy
from io import BytesIO
from pathlib import Path

import cv2
import easyocr
import numpy as np
import requests
import tensorflow as tf
import torch

from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO

## 3. Configuration

In [ ]:
# =========================
# API / MODEL CONFIGURATION
# =========================

PAGES_API_BASE = "https://questions-api-navy.vercel.app/api/pages"
UPLOAD_IMAGE_URL = "https://questions-api-navy.vercel.app/api/upload"
CHAPTER_ID = "dbf7e22d74297ac6bd879949"

MODEL_PATH = "/content/drive/MyDrive/Grad_project-eBook-ML22/Datasets/A_E_dataset10000/model10000_89/best.pt"
IMAGE_TYPE_MODEL_PATH = "/content/drive/MyDrive/LastVersion Model 1/updated_model_finetuned.keras"
HOTSPOT_MODEL_PATH = "/content/drive/MyDrive/The Final/Saved YOLO Models/hotspot/best.pt"
CLASSIFY_CIRCUIT_PATH=tf.keras.models.load_model('/content/drive/MyDrive/LastVersion Model 1/Circuit_classifier.keras')
LOGIC_CIRCUIT_COMPONENT_PATH = "/content/drive/MyDrive/System/models/yolo_models/logic/best.pt"

BASE_OUTPUT_DIR = "/content/extracted_objects"
IMAGES_DIR = "/content/api_pages"
JSON_DIR = "/content/jsons_per_page"

base_dir = Path(BASE_OUTPUT_DIR)
images_dir = Path(IMAGES_DIR)
json_dir = Path(JSON_DIR)
SUBMIT_BLOCKS_URL = "https://questions-api-navy.vercel.app/api/save-blocks"
SUBMIT_TOC_URL = "https://questions-api-navy.vercel.app/api/composite-objects"


## 4. Shared Helper Functions

In [ ]:
IMAGE_LIKE_TYPES = ['image', 'table', 'graph', 'diagram', 'picture','equation', 'quran', 'hadith']

def is_image_type(content_type):
    """تحقق إذا كان النوع يُعامل كصورة"""
    return content_type.lower() in IMAGE_LIKE_TYPES

In [ ]:
def clean_text(s):
    """تنظيف النصوص"""
    if s is None:
        return s
    s = s.replace('\f', '')
    s = re.sub(r'\r', '\n', s)
    s = re.sub(r'\n{3,}', '\n\n', s)
    return s.strip()

In [ ]:
def load_yolov5_model(pt_path):
    """تحميل موديل YOLOv5"""
    model = torch.hub.load('/content/yolov5', 'custom', path=pt_path, source='local')
    print(f"✅ Model loaded. Classes: {model.names}")
    return model

## 5. OCR Helper

In [ ]:
def extract_text_pil(pil_img):
    try:
        arr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        results = reader.readtext(
            arr,
            detail=0,          # بيرجع text بس من غير coordinates
            paragraph=True,    # بيجمع الكلام في paragraphs
            min_size=10,       # يتجاهل حروف صغيرة جداً
            text_threshold=0.7, # ✅ confidence threshold - يتجاهل اللي مش متأكد منه
            allowlist='abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 ,-–'
        )
        text = " ".join(results) if results else ""
        return text.strip()
    except Exception as e:
        print(f"⚠️ OCR Error: {e}")
        return ""

## 6. Image–Caption Matching Helpers

In [ ]:
def create_smart_merged_box(img_coords, cap_coords, padding=10):
    """
    دمج ذكي للـ bounding boxes مع padding محدود

    Args:
        img_coords: إحداثيات الصورة
        cap_coords: إحداثيات الـ caption
        padding: مسافة إضافية حول الـ box (pixels)

    Returns:
        dict: الـ bounding box المدمج
    """
    # الحدود مع padding
    x_min = min(img_coords['x'], cap_coords['x']) - padding
    y_min = min(img_coords['y'], cap_coords['y']) - padding

    x_max = max(
        img_coords['x'] + img_coords['width'],
        cap_coords['x'] + cap_coords['width']
    ) + padding

    y_max = max(
        img_coords['y'] + img_coords['height'],
        cap_coords['y'] + cap_coords['height']
    ) + padding

    return {
        'x': max(0, int(x_min)),  # لا تخرج عن حدود الصورة
        'y': max(0, int(y_min)),
        'width': int(x_max - x_min),
        'height': int(y_max - y_min)
    }


def is_caption_near_image(img_coords, cap_coords, max_gap=120, position_tolerance=0.3):
    """
    تحقق إذا كان الـ caption قريب من الصورة في أي اتجاه
    """
    img_left = img_coords['x']
    img_right = img_coords['x'] + img_coords['width']
    img_top = img_coords['y']
    img_bottom = img_coords['y'] + img_coords['height']
    img_center_y = img_top + img_coords['height'] / 2
    img_center_x = img_left + img_coords['width'] / 2

    cap_left = cap_coords['x']
    cap_right = cap_coords['x'] + cap_coords['width']
    cap_top = cap_coords['y']
    cap_bottom = cap_coords['y'] + cap_coords['height']
    cap_center_y = cap_top + cap_coords['height'] / 2
    cap_center_x = cap_left + cap_coords['width'] / 2

    # حساب التداخل
    h_overlap_left = max(img_left, cap_left)
    h_overlap_right = min(img_right, cap_right)
    h_overlap = max(0, h_overlap_right - h_overlap_left)

    v_overlap_top = max(img_top, cap_top)
    v_overlap_bottom = min(img_bottom, cap_bottom)
    v_overlap = max(0, v_overlap_bottom - v_overlap_top)

    # 1️⃣ Caption أسفل الصورة
    if cap_top >= img_bottom:
        v_gap = cap_top - img_bottom
        if h_overlap > 0 and v_gap <= max_gap:
            return {
                'is_near': True,
                'position': 'below',
                'distance': v_gap,
                'overlap_ratio': h_overlap / min(img_coords['width'], cap_coords['width'])
            }

    # 2️⃣ Caption أعلى الصورة
    elif cap_bottom <= img_top:
        v_gap = img_top - cap_bottom
        if h_overlap > 0 and v_gap <= max_gap:
            return {
                'is_near': True,
                'position': 'above',
                'distance': v_gap,
                'overlap_ratio': h_overlap / min(img_coords['width'], cap_coords['width'])
            }

    # 3️⃣ Caption على اليسار
    elif cap_right <= img_left:
        h_gap = img_left - cap_right
        center_distance = abs(cap_center_y - img_center_y)
        max_center_distance = (img_coords['height'] + cap_coords['height']) * position_tolerance

        if h_gap <= max_gap and center_distance <= max_center_distance:
            return {
                'is_near': True,
                'position': 'left',
                'distance': h_gap,
                'center_alignment': 1 - (center_distance / max_center_distance) if max_center_distance > 0 else 0
            }
        elif h_gap <= max_gap and v_overlap > 0:
            overlap_ratio = v_overlap / min(img_coords['height'], cap_coords['height'])
            if overlap_ratio >= 0.3:
                return {
                    'is_near': True,
                    'position': 'left',
                    'distance': h_gap,
                    'overlap_ratio': overlap_ratio
                }

    # 4️⃣ Caption على اليمين
    elif cap_left >= img_right:
        h_gap = cap_left - img_right
        center_distance = abs(cap_center_y - img_center_y)
        max_center_distance = (img_coords['height'] + cap_coords['height']) * position_tolerance

        if h_gap <= max_gap and center_distance <= max_center_distance:
            return {
                'is_near': True,
                'position': 'right',
                'distance': h_gap,
                'center_alignment': 1 - (center_distance / max_center_distance) if max_center_distance > 0 else 0
            }
        elif h_gap <= max_gap and v_overlap > 0:
            overlap_ratio = v_overlap / min(img_coords['height'], cap_coords['height'])
            if overlap_ratio >= 0.3:
                return {
                    'is_near': True,
                    'position': 'right',
                    'distance': h_gap,
                    'overlap_ratio': overlap_ratio
                }

    # 5️⃣ متداخل
    elif h_overlap > 0 and v_overlap > 0:
        return {
            'is_near': True,
            'position': 'overlapping',
            'distance': 0,
            'overlap_ratio': (h_overlap * v_overlap) / min(
                img_coords['width'] * img_coords['height'],
                cap_coords['width'] * cap_coords['height']
            )
        }

    return {
        'is_near': False,
        'position': None,
        'distance': float('inf')
    }


def merge_images_with_captions_smart(data, max_gap=120, base_dir=None, pil_images_dict=None):
    """
    دمج الصور مع الـ captions بشكل ذكي (كل الاتجاهات)
    """
    from copy import deepcopy

    # تنظيف البيانات
    for item in data:
        if isinstance(item.get('contentType'), str):
            item['contentType'] = item['contentType'].strip().lower()
        if isinstance(item.get('contentValue'), str):
            item['contentValue'] = clean_text(item['contentValue'])

    # فصل الصور والـ captions حسب الصفحة
    pages_data = {}
    for idx, item in enumerate(data):
        page = item.get('page', 0)
        if page not in pages_data:
            pages_data[page] = {'images': [], 'captions': [], 'others': []}

        content_type = item.get('contentType', '')
        if is_image_type(content_type):
            pages_data[page]['images'].append((idx, item))
        elif content_type == 'caption':
            pages_data[page]['captions'].append((idx, item))
        else:
            pages_data[page]['others'].append((idx, item))

    merged_indices = set()
    result = []

    # معالجة كل صفحة
    for page, page_items in pages_data.items():
        images = page_items['images']
        captions = page_items['captions']

        for img_idx, img_item in images:
            if img_idx in merged_indices:
                continue

            img_coords = img_item.get('coordinates')
            if not img_coords:
                result.append(deepcopy(img_item))
                merged_indices.add(img_idx)
                continue

            # البحث عن أفضل caption
            best_caption = None
            best_caption_idx = None
            best_score = -1
            best_position_info = None

            for cap_idx, cap_item in captions:
                if cap_idx in merged_indices:
                    continue

                cap_coords = cap_item.get('coordinates')
                if not cap_coords:
                    continue

                proximity_info = is_caption_near_image(
                    img_coords,
                    cap_coords,
                    max_gap=max_gap,
                    position_tolerance=0.4
                )

                if proximity_info['is_near']:
                    score = 1000 - proximity_info['distance']

                    if proximity_info['position'] == 'below':
                        score += 100

                    if 'overlap_ratio' in proximity_info:
                        score += proximity_info['overlap_ratio'] * 50

                    if 'center_alignment' in proximity_info:
                        score += proximity_info['center_alignment'] * 50

                    if score > best_score:
                        best_score = score
                        best_caption = cap_item
                        best_caption_idx = cap_idx
                        best_position_info = proximity_info

            if best_caption and best_position_info:
                cap_coords = best_caption['coordinates']

                # دمج الـ boxes
                merged_box = create_smart_merged_box(img_coords, cap_coords, padding=10)

                merged_item = {

                        "contentType": img_item.get('contentType', ''),
                        "coordinates": merged_box,
                        "image": img_item.get('image') or img_item.get('imageUrl'),  # ← نفس الـ URL للصور بدون caption
                        "imageCoordinates": img_coords,
                        "imageOcr":img_item.get('imageOcr', ''),
                        "captionOcr": best_caption.get('contentValue', ''),
                        "captionCoordinates": cap_coords,
                        "objectCategory": 'X',
                        "objectType": img_item.get('objectType', ''),
                        "objectId": "",
                        "hotspotMatches": img_item.get('hotspotMatches') or []
                }

                # حفظ الصورة المدمجة
                if base_dir and pil_images_dict and page in pil_images_dict:
                    try:
                        page_img = pil_images_dict[page]
                        merged_crop = page_img.crop((
                            merged_box['x'],
                            merged_box['y'],
                            merged_box['x'] + merged_box['width'],
                            merged_box['y'] + merged_box['height']
                        ))

                        merged_img_fname = f"page_{page:03d}_merged_{img_idx}_{best_caption_idx}.png"
                        merged_img_path = base_dir / 'merged_images' / merged_img_fname
                        merged_crop.save(merged_img_path)

                        # 🔥 رفع الصورة المدمجة
                        print(f"    📤 Uploading merged image...")
                        merged_uploaded_url = upload_image_to_api(merged_img_path, UPLOAD_IMAGE_URL)

                        if merged_uploaded_url:
                            print(f"    ✅ Merged image uploaded: {merged_uploaded_url[:60]}...")
                            merged_item['contentValue'] = merged_uploaded_url  # 🔥 URL المدمجة
                            #merged_item['mergedImagePath'] = str(merged_img_path)  # المسار المحلي
                        else:
                            print(f"    ⚠️ Merged image upload failed")
                            merged_item['mergedImagePath'] = str(merged_img_path)

                    except Exception as e:
                        print(f"    ⚠️ Failed to save/upload merged image: {e}")
                        merged_item['mergedImagePath'] = str(merged_img_path)

                result.append(merged_item)
                merged_indices.add(img_idx)
                merged_indices.add(best_caption_idx)

                pos = best_position_info['position']
                dist = best_position_info['distance']
                print(f"  ✅ Merged image with caption ({pos}, {dist:.0f}px) on page {page}")
            else:
                result.append(deepcopy(img_item))
                merged_indices.add(img_idx)

        # أضف الـ captions المتبقية
        for cap_idx, cap_item in captions:
            if cap_idx not in merged_indices:
                result.append(deepcopy(cap_item))
                merged_indices.add(cap_idx)

        # أضف باقي العناصر
        for other_idx, other_item in page_items['others']:
            if other_idx not in merged_indices:
                result.append(deepcopy(other_item))
                merged_indices.add(other_idx)

    return result

## 7. Image Upload Helper

In [ ]:
def upload_image_to_api(image_path, upload_url):
    """
    رفع صورة إلى API وإرجاع URL
    """
    if not Path(image_path).exists():
        print(f"Image not found: {image_path}")
        return None

    try:
        with open(image_path, 'rb') as f:
            files = {
                'file': (Path(image_path).name, f, 'image/png')
            }

            response = requests.post(upload_url, files=files, timeout=30)
            response.raise_for_status()

            upload_result = response.json()
            print(upload_result)

            # الـ API الحالي قد يرجع URL كسلسلة نصية مباشرة،
            # أو قد يرجع dict يحتوي على الرابط.
            if isinstance(upload_result, str):
                image_url = upload_result
            elif isinstance(upload_result, dict):
                image_url = (
                    upload_result.get('url')
                    or upload_result.get('imageUrl')
                    or upload_result.get('secure_url')
                    or (upload_result.get('data') or {}).get('url')
                )
            else:
                image_url = None

            if image_url:
                return image_url

            print(f"Upload response doesn't contain a usable URL: {upload_result}")
            return None

    except requests.exceptions.Timeout:
        print(f"⏱️ Upload timeout for {Path(image_path).name}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Upload failed for {Path(image_path).name}: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error uploading {Path(image_path).name}: {e}")
        return None

## 8. Scientific Image Prediction Helper

In [ ]:
# ==========================================
# PRIVATE HELPER FOR SCIENTIFIC IMAGE MODEL
# ==========================================

CLASS_NAMES = {
    0: "Bar Chart",
    1: "Chemical equation",
    2: "LifeCycle",
    3: "Numeric Table",
    4: "Pie Chart",
    5: "Unknown",
    6: "circuits",
    7: "classification",
    8: "hotspot_image",
    9: "image_juxtaposition"
}

def _predict_scientific_image_type(image_url, model, target_size=(299, 299)):
    """
    Internal helper used by classify_image().

    It downloads one image, applies the same preprocessing used by
    the trained scientific-image classifier, then returns:
        (class_name, confidence)
    """

    try:
        response = requests.get(image_url, timeout=15)
        response.raise_for_status()

        img = Image.open(BytesIO(response.content)).convert("RGB")
        img = img.resize(target_size)

        img_array = np.array(img, dtype=np.float32)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = img_array / 255.0

        predictions = model.predict(img_array, verbose=0)

        predicted_class = int(np.argmax(predictions, axis=1)[0])
        confidence = float(predictions[0][predicted_class])

        class_name = CLASS_NAMES.get(predicted_class, "Unknown")

        return class_name, confidence

    except Exception as e:
        print(f"Classification error: {e}")
        return "Unknown", 0.0

## 9. Main Function 3 — Process Hotspot Image

This section restores the hotspot logic from the pre-API version while keeping the organized architecture.

- The hotspot model is preloaded once.
- `process_hotspot_image()` runs only for images classified as `hotspot_image`.
- The original `spot` ↔ `spot_text` one-to-one matching logic is preserved.
- The returned payload keeps the previous hotspot JSON structure.


In [ ]:
# ==========================================================
# HOTSPOT HELPERS + API-READY HOTSPOT FUNCTION
# ==========================================================

HOTSPOT_LABEL_MAP = {
    0: "picture",
    1: "spot",
    2: "spot_text"
}


def extract_text_smart(image, spot_center, label_bbox):
    """
    OCR helper restored from the pre-API code.

    image:
        OpenCV BGR image.
    spot_center:
        Kept in the signature for compatibility with the old pipeline.
    label_bbox:
        (x1, y1, x2, y2) of the detected spot_text box.
    """
    try:
        x1, y1, x2, y2 = label_bbox

        padding = 5
        x1 = max(0, int(x1) - padding)
        y1 = max(0, int(y1) - padding)
        x2 = min(image.shape[1], int(x2) + padding)
        y2 = min(image.shape[0], int(y2) + padding)

        roi = image[y1:y2, x1:x2]
        if roi.size == 0:
            return ""

        # Same x4 enlargement used in the old code.
        scale = 4
        roi = cv2.resize(
            roi,
            (roi.shape[1] * scale, roi.shape[0] * scale),
            interpolation=cv2.INTER_CUBIC
        )

        # Background/contrast enhancement using LAB + CLAHE.
        lab = cv2.cvtColor(roi, cv2.COLOR_BGR2LAB)
        l_channel = lab[:, :, 0]

        clahe = cv2.createCLAHE(
            clipLimit=3.0,
            tileGridSize=(8, 8)
        )
        enhanced = clahe.apply(l_channel)

        results_text = []

        # Attempt 1: Otsu binary.
        _, binary1 = cv2.threshold(
            enhanced,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        results_text.append(
            clean_text(
                extract_text_pil(
                    Image.fromarray(binary1)
                )
            )
        )

        # Attempt 2: inverted Otsu.
        _, binary2 = cv2.threshold(
            enhanced,
            0,
            255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )
        results_text.append(
            clean_text(
                extract_text_pil(
                    Image.fromarray(binary2)
                )
            )
        )

        # Attempt 3: adaptive threshold.
        adaptive = cv2.adaptiveThreshold(
            enhanced,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            11,
            2
        )
        results_text.append(
            clean_text(
                extract_text_pil(
                    Image.fromarray(adaptive)
                )
            )
        )

        # Attempt 4: grayscale + Otsu.
        gray = cv2.cvtColor(
            roi,
            cv2.COLOR_BGR2GRAY
        )
        _, binary4 = cv2.threshold(
            gray,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        results_text.append(
            clean_text(
                extract_text_pil(
                    Image.fromarray(binary4)
                )
            )
        )

        results_text = [
            text
            for text in results_text
            if text and len(text) > 1
        ]

        if results_text:
            return max(
                results_text,
                key=len
            )

        return ""

    except Exception as e:
        print(f"OCR error: {e}")
        return ""


def detect_line_between_points(
    image,
    point1,
    point2,
    thickness_tolerance=15
):
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    edges = cv2.Canny(
        gray,
        50,
        150,
        apertureSize=3
    )

    mask = np.zeros_like(edges)

    cv2.line(
        mask,
        point1,
        point2,
        255,
        thickness_tolerance
    )

    intersection = cv2.bitwise_and(
        edges,
        mask
    )

    line_pixels = np.sum(
        intersection > 0
    )

    distance = np.sqrt(
        (point2[0] - point1[0]) ** 2 +
        (point2[1] - point1[1]) ** 2
    )

    return (
        line_pixels / distance
        if distance > 0
        else 0
    )


def point_to_line_distance(
    point,
    line_start,
    line_end
):
    px, py = point
    x1, y1 = line_start
    x2, y2 = line_end

    numerator = abs(
        (y2 - y1) * px
        - (x2 - x1) * py
        + x2 * y1
        - y2 * x1
    )

    denominator = np.sqrt(
        (y2 - y1) ** 2 +
        (x2 - x1) ** 2
    )

    return (
        numerator / denominator
        if denominator > 0
        else float("inf")
    )


def get_point_position_on_line(
    point,
    line_start,
    line_end
):
    px, py = point
    x1, y1 = line_start
    x2, y2 = line_end

    dx = x2 - x1
    dy = y2 - y1

    if dx == 0 and dy == 0:
        return 0

    return (
        ((px - x1) * dx + (py - y1) * dy)
        / (dx * dx + dy * dy)
    )


def detect_connecting_line_hough(
    image,
    spot,
    label
):
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    edges = cv2.Canny(
        gray,
        50,
        150,
        apertureSize=3
    )

    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=50,
        minLineLength=30,
        maxLineGap=10
    )

    if lines is None:
        return 0

    spot_pt = spot["center"]
    label_pt = label["center"]

    max_score = 0

    for line in lines:
        x1, y1, x2, y2 = line[0]

        dist_spot = point_to_line_distance(
            spot_pt,
            (x1, y1),
            (x2, y2)
        )

        dist_label = point_to_line_distance(
            label_pt,
            (x1, y1),
            (x2, y2)
        )

        if (
            dist_spot < 20
            and dist_label < 20
        ):
            score = 1 / (
                dist_spot
                + dist_label
                + 1
            )

            max_score = max(
                max_score,
                score
            )

    return max_score


def match_spots_to_labels_one_to_one(
    spots,
    labels,
    image
):
    """
    Same scoring/matching strategy used in the old hotspot pipeline.
    """

    score_matrix = []

    for spot_idx, spot in enumerate(spots):

        for label_idx, label in enumerate(labels):

            spot_pt = spot["center"]
            label_pt = label["center"]

            distance = np.sqrt(
                (spot_pt[0] - label_pt[0]) ** 2 +
                (spot_pt[1] - label_pt[1]) ** 2
            )

            line_score = (
                detect_line_between_points(
                    image,
                    spot_pt,
                    label_pt
                )
            )

            hough_score = (
                detect_connecting_line_hough(
                    image,
                    spot,
                    label
                )
            )

            dx = label_pt[0] - spot_pt[0]
            dy = label_pt[1] - spot_pt[1]

            angle = np.arctan2(
                dy,
                dx
            )

            direction_score = abs(
                np.cos(angle)
            )

            crossing_penalty = 1.0

            for other_spot in spots:

                if other_spot == spot:
                    continue

                line_dist = (
                    point_to_line_distance(
                        other_spot["center"],
                        spot_pt,
                        label_pt
                    )
                )

                if line_dist < 10:

                    t = (
                        get_point_position_on_line(
                            other_spot["center"],
                            spot_pt,
                            label_pt
                        )
                    )

                    if 0.1 < t < 0.9:
                        crossing_penalty = 0.5
                        break

            spatial_score = (
                1.2
                if abs(dx) > abs(dy)
                else 1.0
            )

            final_score = (
                (1000 / (distance + 1)) * 0.10
                + line_score * 100 * 0.40
                + hough_score * 100 * 0.30
                + direction_score * 0.10
                + crossing_penalty * 0.05
                + spatial_score * 0.05
            )

            score_matrix.append({
                "spot_idx": spot_idx,
                "label_idx": label_idx,
                "score": final_score,
                "distance": distance,
                "line_score": line_score,
                "hough_score": hough_score,
                "direction_score": direction_score
            })

    score_matrix.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    matched_spots = set()
    matched_labels = set()
    matches = []

    for pair in score_matrix:

        spot_idx = pair["spot_idx"]
        label_idx = pair["label_idx"]

        if (
            spot_idx not in matched_spots
            and label_idx not in matched_labels
        ):
            matches.append({
                "spot": spots[spot_idx],
                "label": labels[label_idx],
                "score": pair["score"],
                "distance": pair["distance"],
                "line_score": pair["line_score"],
                "hough_score": pair["hough_score"],
                "direction_score": pair["direction_score"]
            })

            matched_spots.add(
                spot_idx
            )

            matched_labels.add(
                label_idx
            )

            if (
                len(matched_spots) == len(spots)
                or len(matched_labels) == len(labels)
            ):
                break

    return (
        matches,
        matched_spots,
        matched_labels
    )


def _process_hotspot_cv2(
    image,
    hotspot_model,
    uploaded_url="",
    image_ocr_text=""
):
    """
    Internal hotspot processor.

    Difference from the old version:
    the YOLO hotspot model is passed already loaded instead of
    being loaded again for every image.
    """

    try:
        results = hotspot_model(
            image,
            verbose=False
        )

        spots = []
        labels = []

        for result in results:

            for box in result.boxes:

                cls = int(
                    box.cls[0]
                )

                conf = float(
                    box.conf[0]
                )

                x1, y1, x2, y2 = (
                    box.xyxy[0]
                    .cpu()
                    .numpy()
                )

                entry = {
                    "center": (
                        int((x1 + x2) / 2),
                        int((y1 + y2) / 2)
                    ),
                    "bbox": (
                        int(x1),
                        int(y1),
                        int(x2),
                        int(y2)
                    ),
                    "conf": conf
                }

                detected_label = (
                    HOTSPOT_LABEL_MAP.get(
                        cls,
                        "unknown"
                    )
                )

                if detected_label == "spot":
                    spots.append(
                        entry
                    )

                elif detected_label == "spot_text":
                    labels.append(
                        entry
                    )

        print(
            f"  📍 Found {len(spots)} spots "
            f"and {len(labels)} labels"
        )

        if not spots or not labels:

            print(
                "  ⚠️ No spots or labels found!"
            )

            return None

        matches, _, _ = (
            match_spots_to_labels_one_to_one(
                spots,
                labels,
                image
            )
        )

        print(
            f"  ✅ Hotspot matching: "
            f"{len(matches)} pairs matched"
        )

        hotspots_list = []

        for i, match in enumerate(matches):

            spot_pt = (
                match["spot"]["center"]
            )

            label_bbox = (
                match["label"]["bbox"]
            )

            label_text = (
                extract_text_smart(
                    image,
                    spot_pt,
                    label_bbox
                )
            )

            if label_text:

                label_text = "".join(
                    char
                    for char in label_text
                    if (
                        char.isalnum()
                        or char.isspace()
                        or char in [
                            "-",
                            "_",
                            ","
                        ]
                    )
                )

                label_text = (
                    label_text.strip()
                )

                words = (
                    label_text.split()
                )

                words = [
                    word
                    for word in words
                    if (
                        len(word) > 1
                        or word.lower()
                        in ["a", "i"]
                    )
                ]

                label_text = " ".join(
                    words
                )

            if (
                not label_text
                or len(label_text) < 2
            ):
                label_text = (
                    f"Label_{i + 1}"
                )

            print(
                f"  ✓ Spot #{i + 1}: "
                f"'{label_text}'"
            )

            hotspots_list.append({
                "_Xposition_": str(
                    spot_pt[0]
                ),
                "_Yposition_": str(
                    spot_pt[1]
                ),
                "_Header_": {
                    "text": label_text,
                    "x1": str(
                        label_bbox[0]
                    ),
                    "y1": str(
                        label_bbox[1]
                    ),
                    "x2": str(
                        label_bbox[2]
                    ),
                    "y2": str(
                        label_bbox[3]
                    )
                },
                "_HotspotText_": "",
                "_HotspotText2_": ""
            })

        return {
            "_Picture_": uploaded_url,
            "_AltText_": image_ocr_text,
            "hotSpots": hotspots_list
        }

    except Exception as e:

        print(
            f"  ❌ Hotspot processing failed: {e}"
        )

        traceback.print_exc()

        return None


# ==========================================================
# MAIN FUNCTION 3
# process_hotspot_image()
# ==========================================================

def process_hotspot_image(
    image_url,
    hotspot_model,
    image_ocr_text=""
):
    """
    API-ready hotspot processing service.

    Call this function only after classify_image() returns:
        imageType == "hotspot_image"

    INPUT
    -----
    image_url : str
        URL of ONE extracted scientific image.

    hotspot_model
        Preloaded Ultralytics YOLO hotspot model.

    image_ocr_text : str, optional
        OCR text already extracted from the image.

    OUTPUT
    ------
    The same hotspot payload used by the old pipeline:
    {
        "_Picture_": "...",
        "_AltText_": "...",
        "hotSpots": [
            {
                "_Xposition_": "...",
                "_Yposition_": "...",
                "_Header_": {
                    "text": "...",
                    "x1": "...",
                    "y1": "...",
                    "x2": "...",
                    "y2": "..."
                },
                "_HotspotText_": "",
                "_HotspotText2_": ""
            }
        ]
    }
    """

    if (
        not isinstance(image_url, str)
        or not image_url.strip()
    ):
        raise ValueError(
            "image_url must be a non-empty string."
        )

    image_url = (
        image_url.strip()
    )

    response = requests.get(
        image_url,
        timeout=30
    )

    response.raise_for_status()

    pil_image = (
        Image.open(
            BytesIO(
                response.content
            )
        )
        .convert("RGB")
    )

    image_cv2 = cv2.cvtColor(
        np.array(pil_image),
        cv2.COLOR_RGB2BGR
    )

    return _process_hotspot_cv2(
        image=image_cv2,
        hotspot_model=hotspot_model,
        uploaded_url=image_url,
        image_ocr_text=image_ocr_text
    )


## 10. Load Models Once


In [ ]:
# ==========================================
# LOAD MODELS ONCE
# ==========================================
# Important for the future API:
# Models are loaded once when the service starts,
# NOT inside the endpoint/function on every request.

print("🔄 Loading OCR and models...")

reader = easyocr.Reader(["en"])

OBJECT_DETECTOR = load_yolov5_model(
    MODEL_PATH
)

IMAGE_CLASSIFIER = tf.keras.models.load_model(
    IMAGE_TYPE_MODEL_PATH
)

HOTSPOT_MODEL = YOLO(
    HOTSPOT_MODEL_PATH
)

print("✅ OCR, object detector, image classifier, and hotspot model are ready.")

🔄 Loading OCR and models...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteYOLOv5 🚀 v7.0-552-g35b48237 Python-3.13.15 torch-2.11.0+cpu CPU

Fusing layers... 
Model summary: 220 layers, 35,312,412 parameters, 0 gradients, 49.0 GFLOPs
Adding AutoShape... 
✅ Model loaded. Classes: {0: 'section', 1: 'text', 2: 'image', 3: 'diagram', 4: 'graph', 5: 'table', 6: 'table of contents', 7: 'references', 8: 'index', 9: 'title', 10: 'caption', 11: 'header'}
✅ OCR, object detector, image classifier, and hotspot model are ready.


## 11. Main Function 1 — Extract Objects from eBook


In [ ]:
# ==========================================================
# MAIN FUNCTION 1
# extract_objects_from_ebook()
# ==========================================================

def extract_objects_from_ebook(chapter_id, detector):
    """
    API-ready eBook object extraction service.

    INPUT
    -----
    chapter_id : str
        Chapter identifier received from the future API request.

    detector
        Preloaded YOLOv5 object detector.

    OUTPUT
    ------
    A JSON-serializable Python dictionary:
    {
        "chapterId": "...",
        "pages": [
            {
                "pageId": "...",
                "pageIndex": 0,
                "pageWidth": 1200,
                "pageHeight": 1600,
                "objects": [...]
            }
        ]
    }

    IMPORTANT:
    - This function performs object extraction only.
    - It does NOT classify generic scientific images.
    - It does NOT generate the TOC.
    - It does NOT submit final blocks to another endpoint.
    """

    # -----------------------------
    # Validate API-style input
    # -----------------------------
    if not isinstance(chapter_id, str) or not chapter_id.strip():
        raise ValueError("chapter_id must be a non-empty string.")

    chapter_id = chapter_id.strip()

    # -----------------------------
    # Prepare local working folders
    # -----------------------------
    base_dir = Path(BASE_OUTPUT_DIR)
    images_dir = Path(IMAGES_DIR)
    json_dir = Path(JSON_DIR)

    for directory in [base_dir, images_dir, json_dir]:
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True, exist_ok=True)

    for subfolder in [
        "images", "texts", "captions", "sections",
        "subsections", "merged_images", "titles",
        "headers", "boxes"
    ]:
        (base_dir / subfolder).mkdir(exist_ok=True)

    # -----------------------------
    # Retrieve eBook pages
    # -----------------------------
    print(f"🔄 Fetching pages for chapter: {chapter_id}")

    response = requests.get(
        PAGES_API_BASE,
        params={"chapterId": chapter_id},
        timeout=60
    )
    response.raise_for_status()

    pages = response.json()

    if not pages:
        return {
            "chapterId": chapter_id,
            "pages": []
        }

    print(f"✅ Found {len(pages)} pages")
    print("Response type:", type(pages))
    print("Response content:", pages)

    extracted_pages = []
    pil_images_dict = {}

    # ======================================================
    # PAGE LOOP
    # ======================================================
    for page_idx, page in enumerate(
        tqdm(pages, desc="Extracting page objects")
    ):
        page_objects = []

        page_id = (
            page.get("_id")
            or page.get("id")
            or page.get("pageId")
        )

        image_url = page.get("url")

        if not image_url:
            print(f"⚠️ Page {page_idx}: missing image URL")
            continue

        page_image_path = (
            Path(IMAGES_DIR) /
            f"page_{page_idx:03d}.png"
        )

        try:
            # -----------------------------
            # Download page image
            # -----------------------------
            page_response = requests.get(
                image_url,
                stream=True,
                timeout=60
            )
            page_response.raise_for_status()

            with open(page_image_path, "wb") as f:
                shutil.copyfileobj(page_response.raw, f)

            # -----------------------------
            # YOLO object extraction
            # -----------------------------
            results = detector(str(page_image_path))

            detections = (
                results
                .pandas()
                .xyxy[0]
                .to_dict(orient="records")
            )

            page_image = Image.open(
                page_image_path
            ).convert("RGB")

            page_width, page_height = page_image.size
            pil_images_dict[page_idx] = page_image

            # ==================================================
            # DETECTED OBJECT LOOP
            # ==================================================
            for det_idx, detection in enumerate(detections):

                xmin = int(detection["xmin"])
                ymin = int(detection["ymin"])
                xmax = int(detection["xmax"])
                ymax = int(detection["ymax"])

                label = detection.get(
                    "name",
                    str(detection.get("class"))
                )

                detection_confidence = float(
                    detection.get("confidence", 0.0)
                )

                content_type = label.lower().strip()

                if content_type == "text":
                    content_type = "paragraph"

                coordinates = {
                    "x": xmin,
                    "y": ymin,
                    "width": xmax - xmin,
                    "height": ymax - ymin
                }

                crop = page_image.crop(
                    (xmin, ymin, xmax, ymax)
                )

                # ==============================================
                # IMAGE-LIKE OBJECT
                # ==============================================
                if is_image_type(content_type):

                    folder_name = content_type + "s"

                    image_name = (
                        f"page_{page_idx:03d}_"
                        f"{content_type}_"
                        f"{det_idx}.png"
                    )

                    extracted_image_path = (
                        Path(BASE_OUTPUT_DIR) /
                        folder_name /
                        image_name
                    )

                    extracted_image_path.parent.mkdir(
                        parents=True,
                        exist_ok=True
                    )

                    crop.save(extracted_image_path)

                    uploaded_url = upload_image_to_api(
                        extracted_image_path,
                        UPLOAD_IMAGE_URL
                    )

                    image_ocr = extract_text_pil(crop)

                    obj = {
                        "contentType": content_type,
                        "contentValue": uploaded_url or "",
                        "coordinates": coordinates,

                        "image": uploaded_url or "",
                        "imageCoordinates": coordinates.copy(),
                        "imageOcr": image_ocr,

                        "captionOcr": "",
                        "captionCoordinates": {},

                        "objectCategory": content_type,

                        # No scientific image classification here.
                        # Generic image stays "image" until API 2.
                        "objectType": content_type,

                        "objectId": "",
                        "classificationConfidence": None,
                        "detectionConfidence": detection_confidence,
                        "page": page_idx,
                        "hotspotMatches": []
                    }

                # ==============================================
                # CAPTION
                # ==============================================
                elif content_type == "caption":

                    text = extract_text_pil(crop)

                    obj = {
                        "contentType": "caption",
                        "contentValue": clean_text(text),
                        "coordinates": coordinates,
                        "objectCategory": "",
                        "objectType": "",
                        "detectionConfidence": detection_confidence,
                        "page": page_idx
                    }

                # ==============================================
                # TEXT / SECTION / SUBSECTION / OTHER
                # ==============================================
                else:

                    text = extract_text_pil(crop)

                    obj = {
                        "contentType": content_type,
                        "contentValue": clean_text(text),
                        "coordinates": coordinates,
                        "objectCategory": "",
                        "objectType": "",
                        "detectionConfidence": detection_confidence,
                        "page": page_idx
                    }

                page_objects.append(obj)

            # -----------------------------
            # Image-caption matching
            # -----------------------------
            merged_objects = merge_images_with_captions_smart(
                page_objects,
                max_gap=120,
                base_dir=Path(BASE_OUTPUT_DIR),
                pil_images_dict=pil_images_dict
            )

            # -----------------------------
            # JSON-friendly page result
            # -----------------------------
            extracted_pages.append({
                "pageId": page_id,
                "pageIndex": page_idx,
                "pageWidth": page_width,
                "pageHeight": page_height,
                "objects": merged_objects
            })

        except Exception as e:
            print(f"⚠️ Error processing page {page_idx}: {e}")
            traceback.print_exc()

    result = {
        "chapterId": chapter_id,
        "pages": extracted_pages
    }

    # Optional local debug copy.
    # This does not affect the API response contract.
    debug_path = (
        Path(BASE_OUTPUT_DIR) /
        "extracted_ebook_objects.json"
    )

    with open(debug_path, "w", encoding="utf-8") as f:
        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"\n✅ Extraction complete. "
        f"Processed pages: {len(extracted_pages)}"
    )

    return result

## 12. Main Function 2 — Classify One Image


In [ ]:
# ==========================================================
# MAIN FUNCTION 2
# classify_image()
# ==========================================================

def classify_image(image_url, classifier):
    """
    API-ready scientific image classification service.

    INPUT
    -----
    image_url : str
        URL of ONE extracted scientific image.

    classifier
        Preloaded scientific-image classification model.

    OUTPUT
    ------
    JSON-serializable dictionary:
    {
        "imageUrl": "...",
        "imageType": "Bar Chart",
        "confidence": 0.95
    }

    IMPORTANT:
    - One API request = one image.
    - The model is passed preloaded and is NOT loaded here.
    """

    if not isinstance(image_url, str) or not image_url.strip():
        raise ValueError("image_url must be a non-empty string.")

    image_url = image_url.strip()

    predicted_type, confidence = (
        _predict_scientific_image_type(
            image_url,
            classifier
        )
    )

    return {
        "imageUrl": image_url,
        "imageType": predicted_type,
        "confidence": round(float(confidence), 6)
    }

## 13. Colab Routing Helper

This helper is only orchestration for the current Colab run. It classifies each extracted generic image and invokes `process_hotspot_image()` only when the predicted type is `hotspot_image`.


In [ ]:
# ==========================================================
# COLAB ROUTING HELPER
# Classification -> optional hotspot processing
# ==========================================================

def classify_all_extracted_images(
    extraction_result,
    classifier,
    hotspot_model
):
    """
    Colab orchestration helper only.

    Flow for each generic image:
        classify_image()
            |
            +-- hotspot_image
                    |
                    +-- process_hotspot_image()

    This keeps classification and hotspot processing as
    separate API-ready functions.
    """

    if not extraction_result:
        return extraction_result

    classified = 0
    hotspot_processed = 0

    for page in extraction_result.get(
        "pages",
        []
    ):

        for obj in page.get(
            "objects",
            []
        ):

            content_type = (
                obj
                .get(
                    "contentType",
                    ""
                )
                .lower()
                .strip()
            )

            # Same rule as the old pipeline:
            # only generic "image" objects enter the
            # scientific image classifier.
            if content_type != "image":
                continue

            image_url = (
                obj.get("image")
                or obj.get("contentValue")
            )

            if not image_url:
                continue

            result = classify_image(
                image_url,
                classifier
            )

            detected_type = (
                result["imageType"]
            )

            obj["objectType"] = (
                detected_type
            )

            obj[
                "classificationConfidence"
            ] = result["confidence"]

            classified += 1

            # ==============================================
            # HOTSPOT ROUTING
            # ==============================================
            if (
                detected_type
                and detected_type.lower()
                == "hotspot_image"
            ):

                print(
                    "  🎯 Hotspot detected! "
                    "Running hotspot processing..."
                )

                hotspot_result = (
                    process_hotspot_image(
                        image_url=image_url,
                        hotspot_model=hotspot_model,
                        image_ocr_text=obj.get(
                            "imageOcr",
                            ""
                        )
                    )
                )

                obj["hotspotMatches"] = (
                    hotspot_result
                    if hotspot_result
                    else []
                )

                hotspot_processed += 1

            else:

                obj["hotspotMatches"] = []

    print(
        f"✅ Classified {classified} extracted images."
    )

    print(
        f"✅ Processed {hotspot_processed} hotspot images."
    )

    return extraction_result


## 14. Main Function 4 — Generate TOC


In [ ]:
# ==========================================================
# MAIN FUNCTION 4
# generate_toc()
# ==========================================================

def generate_toc(extraction_result):
    """
    API-ready TOC generation service.

    INPUT
    -----
    extraction_result : dict
        JSON returned by extract_objects_from_ebook().

    OUTPUT
    ------
    JSON-serializable dictionary:
    {
        "name": "",
        "type": "TOC",
        "chapterId": "...",
        "blocks": [
            {
                "contentType": "Section",
                "contentValue": "...",
                "pageId": "..."
            }
        ]
    }

    IMPORTANT:
    - This function uses only Section and SubSection objects.
    - It does NOT depend on the scientific image classifier.
    """

    if not isinstance(extraction_result, dict):
        raise ValueError(
            "extraction_result must be a dictionary."
        )

    chapter_id = extraction_result.get("chapterId")

    if not chapter_id:
        raise ValueError(
            "extraction_result does not contain chapterId."
        )

    toc_blocks = []
    last_page_id = None

    for page in extraction_result.get("pages", []):

        page_id = page.get("pageId")
        last_page_id = page_id

        for obj in page.get("objects", []):

            content_type = (
                obj
                .get("contentType", "")
                .lower()
                .strip()
            )

            if content_type not in [
                "section",
                "subsection"
            ]:
                continue

            content_value = (
                obj
                .get("contentValue", "")
                .strip()
            )

            if not content_value:
                continue

            toc_blocks.append({
                "contentType": (
                    "Section"
                    if content_type == "section"
                    else "SubSection"
                ),
                "contentValue": content_value,
                "pageId": page_id
            })

    return {
        "name": "",
        "type": "TOC",
        "chapterId": chapter_id,
        "blocks": toc_blocks,
        "pageId": last_page_id
    }

## Site Submission Layer

The four main functions remain independent and API-ready. These helpers only submit the processed result to the current website so you can see it while testing in Colab.

In [ ]:
# # ==========================================================
# # SITE SUBMISSION LAYER
# # ==========================================================
# # These helpers are not new APIs. They only connect the
# # three API-ready functions to the current website during
# # Colab testing.


# def convert_to_api_format(page_objs, page_id, page_width, page_height):
#     blocks = []

#     object_type_map = {
#         "Bar Chart": "Bar Chart",
#         "Chemical equation": "Chemical Equation",
#         "LifeCycle": "Lifecycles",
#         "Numeric Table": "Numeric Table",
#         "Pie Chart": "Pie Chart",
#         "circuits": "Chart",
#         "circuits_Electric": "Chart",
#         "circuits_Logic": "Chart",
#         "classification": "Classification",
#         "hotspot_image": "Hotspot Image",
#         "image_juxtaposition": "Image Juxtaposition",
#         "Unknown": "Picture",
#         # "image": "Picture",
#         # "picture": "Picture",
#         "table": "Table",
#         "diagram": "Chart",
#         "graph": "Chart",
#     }

#     illustrative_content_types = [
#         "image",
#         "picture",
#         "image_with_caption"
#     ]

#     for obj in page_objs:
#         coords = obj.get("coordinates", {})
#         content_type = obj.get("contentType", "")
#         content_type_lower = content_type.lower()

#         if content_type_lower in [
#             "section",
#             "subsection",
#             "title",
#             "header"
#         ]:
#             continue

#         x_percent = (coords.get("x", 0) / page_width) * 100 if page_width else 0
#         y_percent = (coords.get("y", 0) / page_height) * 100 if page_height else 0
#         width_percent = (coords.get("width", 0) / page_width) * 100 if page_width else 0
#         height_percent = (coords.get("height", 0) / page_height) * 100 if page_height else 0

#         api_content_type = {
#             "image": "image",
#             "picture": "image",
#             "image_with_caption": "image",
#             "caption": "Caption",
#             "paragraph": "Paragraph",
#             "text": "Paragraph",
#             "table": "Table",
#             "diagram": "Diagram",
#             "equation": "Equation",
#             "quran": "Quran",
#             "hadith": "Hadith",
#             "graph": "Graph",
#             "box": "Box"
#         }.get(content_type_lower, "Paragraph")

#         if is_image_type(content_type) or content_type_lower == "image_with_caption":
#             content_value = (
#                 obj.get("image")
#                 or obj.get("imageUrl")
#                 or obj.get("contentValue")
#                 or ""
#             )
#         else:
#             content_value = obj.get("contentValue", "")

#         detected_type = obj.get("objectType", "")
#         scientific_object_type = object_type_map.get(
#             detected_type,
#             "Picture" if is_image_type(content_type) else ""
#         )

#         item_type = (
#             "Illustrative object"
#             if content_type_lower in illustrative_content_types
#             else "Simple item"
#         )

#         hotspot_data = (
#             obj.get("hotspotMatches")
#             or obj.get("hotspotData")
#             or None
#         )

#         if is_image_type(content_type) or content_type_lower == "image_with_caption":
#             image_data = {
#                 "image": obj.get("image") or obj.get("imageUrl") or "",
#                 "imageCoordinates": obj.get("imageCoordinates", {}),
#                 "captionOcr": obj.get("captionOcr", ""),
#                 "captionCoordinates": obj.get("captionCoordinates", {}),
#             }
#         else:
#             image_data = {}

#         block = {
#             "pageId": page_id,
#             "status": "new",
#             "coordinates": {
#                 "x": round(x_percent, 2),
#                 "y": round(y_percent, 2),
#                 "width": round(width_percent, 2),
#                 "height": round(height_percent, 2),
#                 "unit": "percentage"
#             },
#             # دي المسؤوله عن illustrartive =>>>api content type

#             "contentType": api_content_type,
#             "contentValue": content_value,

#             # Old site behavior
#             "objectType": item_type,
#             "itemType": item_type,

#             "objectCategory": 'X',
#             "objectId": obj.get("objectId", ""),
#             "hotspotData": hotspot_data,

#             # Keep classifier result available for debugging/future API work
#             "scientificObjectType": scientific_object_type,
#             "classificationConfidence": obj.get("classificationConfidence"),

#             **image_data
#         }

#         blocks.append(block)

#     return blocks


# def submit_blocks_to_site(blocks):
#     if not blocks:
#         print("⚠️ No blocks to submit.")
#         return None

#     try:
#         response = requests.post(
#             SUBMIT_BLOCKS_URL,
#             json={"blocks": blocks},
#             headers={"Content-Type": "application/json"},
#             timeout=120
#         )
#         response.raise_for_status()
#         result = response.json()
#         print(f"✅ Submitted {len(blocks)} blocks to site.")
#         return result
#     except Exception as e:
#         print(f"❌ Block submission failed: {e}")
#         return None


# def submit_toc_to_site(toc_result):
#     if not toc_result:
#         print("⚠️ No TOC to submit.")
#         return None

#     try:
#         response = requests.post(
#             SUBMIT_TOC_URL,
#             json=toc_result,
#             headers={"Content-Type": "application/json"},
#             timeout=300
#         )
#         response.raise_for_status()
#         result = response.json()
#         print(f"✅ TOC submitted to site. Entries: {len(toc_result.get('blocks', []))}")
#         return result
#     except Exception as e:
#         print(f"❌ TOC submission failed: {e}")
#         return None


# def classify_all_extracted_images(extraction_result, classifier):
#     """Colab orchestration helper only; not a fourth API."""
#     if not extraction_result:
#         return extraction_result

#     classified = 0
#     for page in extraction_result.get("pages", []):
#         for obj in page.get("objects", []):
#             if obj.get("contentType", "").lower().strip() != "image":
#                 continue

#             image_url = obj.get("image") or obj.get("contentValue")
#             if not image_url:
#                 continue

#             result = classify_image(image_url, classifier)
#             obj["objectType"] = result["imageType"]
#             obj["classificationConfidence"] = result["confidence"]
#             classified += 1

#     print(f"✅ Classified {classified} extracted images.")
#     return extraction_result


# def submit_extraction_result_to_site(extraction_result):
#     if not extraction_result:
#         print("❌ No extraction result available.")
#         return []

#     responses = []
#     for page in extraction_result.get("pages", []):
#         page_id = page.get("pageId")
#         objects = page.get("objects", [])

#         if not page_id or not objects:
#             continue

#         blocks = convert_to_api_format(
#             objects,
#             page_id,
#             page.get("pageWidth", 0),
#             page.get("pageHeight", 0)
#         )

#         print(f"📦 Page {page.get('pageIndex', 0) + 1}: {len(blocks)} blocks")
#         response = submit_blocks_to_site(blocks)
#         responses.append({"pageId": page_id, "response": response})

#     return responses

In [ ]:
def convert_to_api_format(page_objs, page_id, page_width, page_height):

    blocks = []

    # ======================================================
    # Scientific image classification -> Site Content Type
    # ======================================================
    object_type_map = {
        "Bar Chart": "Bar Chart",
        "Chemical equation": "Chemical Equation",
        "LifeCycle": "Lifecycles",
        "Numeric Table": "Numeric Table",
        "Pie Chart": "Pie Chart",

        "circuits": "Chart",
        "circuits_Electric": "Chart",
        "circuits_Logic": "Chart",

        "classification": "Classification",

        # IMPORTANT
        "hotspot_image": "Hotspot Image",

        "image_juxtaposition": "Image Juxtaposition",

        "Unknown": "Picture",

        "table": "Table",
        "diagram": "Chart",
        "graph": "Chart",
    }

    # الصور تعتبر Illustrative Objects
    illustrative_content_types = [
        "image",
        "picture",
        "image_with_caption"
    ]

    # ======================================================
    # LOOP THROUGH PAGE OBJECTS
    # ======================================================
    for obj in page_objs:

        coords = obj.get("coordinates", {})

        content_type = obj.get(
            "contentType",
            ""
        )

        content_type_lower = (
            content_type
            .lower()
            .strip()
        )

        # ==================================================
        # Sections / Titles are handled separately
        # ==================================================
        if content_type_lower in [
            "section",
            "subsection",
            "title",
            "header"
        ]:
            continue

        # ==================================================
        # Coordinates -> percentage
        # ==================================================
        x_percent = (
            coords.get("x", 0) / page_width
        ) * 100 if page_width else 0

        y_percent = (
            coords.get("y", 0) / page_height
        ) * 100 if page_height else 0

        width_percent = (
            coords.get("width", 0) / page_width
        ) * 100 if page_width else 0

        height_percent = (
            coords.get("height", 0) / page_height
        ) * 100 if page_height else 0

        # ==================================================
        # Get classifier result
        # ==================================================
        detected_type = obj.get(
            "objectType",
            ""
        )

        # مثال:
        # hotspot_image -> Hotspot Image
        # Bar Chart     -> Bar Chart
        # Unknown       -> Picture
        scientific_object_type = object_type_map.get(
            detected_type,
            "Picture"
            if content_type_lower in illustrative_content_types
            else ""
        )

        # ==================================================
        # API CONTENT TYPE
        # ==================================================

        # لو IMAGE:
        # استخدم نتيجة الـ Scientific Classifier
        if content_type_lower in illustrative_content_types:

            api_content_type = scientific_object_type

        # لو مش image:
        # استخدم النوع العادي
        else:

            api_content_type = {

                "caption": "Caption",

                "paragraph": "Paragraph",

                "text": "Paragraph",

                "table": "Table",

                "diagram": "Diagram",

                "equation": "Equation",

                "quran": "Quran",

                "hadith": "Hadith",

                "graph": "Graph",

                "box": "Box"

            }.get(
                content_type_lower,
                "Paragraph"
            )

        # ==================================================
        # CONTENT VALUE
        # ==================================================
        if (
            is_image_type(content_type)
            or content_type_lower == "image_with_caption"
        ):

            content_value = (

                obj.get("image")

                or obj.get("imageUrl")

                or obj.get("contentValue")

                or ""
            )

        else:

            content_value = obj.get(
                "contentValue",
                ""
            )

        # ==================================================
        # ITEM TYPE
        # ==================================================
        item_type = (

            "Illustrative object"

            if content_type_lower
            in illustrative_content_types

            else "Simple item"
        )

        # ==================================================
        # HOTSPOT DATA
        # ==================================================
        hotspot_data = (

            obj.get("hotspotMatches")

            or obj.get("hotspotData")

            or None
        )

        # ==================================================
        # IMAGE METADATA
        # ==================================================
        if (
            is_image_type(content_type)
            or content_type_lower == "image_with_caption"
        ):

            image_data = {

                "image": (
                    obj.get("image")
                    or obj.get("imageUrl")
                    or ""
                ),

                "imageCoordinates":
                    obj.get(
                        "imageCoordinates",
                        {}
                    ),

                "captionOcr":
                    obj.get(
                        "captionOcr",
                        ""
                    ),

                "captionCoordinates":
                    obj.get(
                        "captionCoordinates",
                        {}
                    ),
            }

        else:

            image_data = {}

        # ==================================================
        # FINAL BLOCK
        # ==================================================
        block = {

            "pageId": page_id,

            "status": "new",

            "coordinates": {

                "x": round(
                    x_percent,
                    2
                ),

                "y": round(
                    y_percent,
                    2
                ),

                "width": round(
                    width_percent,
                    2
                ),

                "height": round(
                    height_percent,
                    2
                ),

                "unit": "percentage"
            },

            # ==============================================
            # أهم تعديل
            # ==============================================
            "contentType": api_content_type,

            "contentValue": content_value,

            # الموقع يعرف أنها Illustrative object
            "objectType": item_type,

            "itemType": item_type,

            "objectCategory": "X",

            "objectId":
                obj.get(
                    "objectId",
                    ""
                ),

            "hotspotData":
                hotspot_data,

            # نتيجة الـ classifier الأصلية
            "scientificObjectType":
                scientific_object_type,

            "classificationConfidence":
                obj.get(
                    "classificationConfidence"
                ),

            **image_data
        }

        blocks.append(
            block
        )

    return blocks

In [ ]:
# ==========================================================
# ACTIVE COLAB SITE SUBMISSION HELPERS
# ==========================================================

def submit_blocks_to_site(blocks):
    if not blocks:
        print("⚠️ No blocks to submit.")
        return None

    try:
        response = requests.post(
            SUBMIT_BLOCKS_URL,
            json={"blocks": blocks},
            headers={"Content-Type": "application/json"},
            timeout=120
        )

        response.raise_for_status()

        result = response.json()

        print(
            f"✅ Submitted {len(blocks)} blocks to site."
        )

        return result

    except Exception as e:
        print(
            f"❌ Block submission failed: {e}"
        )
        return None


def submit_toc_to_site(toc_result):
    if not toc_result:
        print("⚠️ No TOC to submit.")
        return None

    try:
        response = requests.post(
            SUBMIT_TOC_URL,
            json=toc_result,
            headers={"Content-Type": "application/json"},
            timeout=300
        )

        response.raise_for_status()

        result = response.json()

        print(
            "✅ TOC submitted to site. "
            f"Entries: {len(toc_result.get('blocks', []))}"
        )

        return result

    except Exception as e:
        print(
            f"❌ TOC submission failed: {e}"
        )
        return None


def submit_extraction_result_to_site(
    extraction_result
):
    if not extraction_result:
        print(
            "❌ No extraction result available."
        )
        return []

    responses = []

    for page in extraction_result.get(
        "pages",
        []
    ):

        page_id = page.get(
            "pageId"
        )

        objects = page.get(
            "objects",
            []
        )

        if (
            not page_id
            or not objects
        ):
            continue

        blocks = convert_to_api_format(
            objects,
            page_id,
            page.get(
                "pageWidth",
                0
            ),
            page.get(
                "pageHeight",
                0
            )
        )

        print(
            f"📦 Page "
            f"{page.get('pageIndex', 0) + 1}: "
            f"{len(blocks)} blocks"
        )

        response = (
            submit_blocks_to_site(
                blocks
            )
        )

        responses.append({
            "pageId": page_id,
            "response": response
        })

    return responses


## API Contracts

### API 1 — Extract Objects

Future endpoint example:

`POST /extract-objects`

Request body:

```json
{
  "chapterId": "ed8da075f9f4f1105d53bd06"
}
```

The endpoint will call:

```python
extract_objects_from_ebook(
    request.chapterId,
    OBJECT_DETECTOR
)
```

---

### API 2 — Classify One Image

Future endpoint example:

`POST /classify-image`

Request body:

```json
{
  "imageUrl": "https://example.com/image.png"
}
```

The endpoint will call:

```python
classify_image(
    request.imageUrl,
    IMAGE_CLASSIFIER
)
```

If the response contains:

```json
{
  "imageType": "hotspot_image"
}
```

the orchestration layer can route the same image to API 3.

---

### API 3 — Process Hotspot Image

Future endpoint example:

`POST /process-hotspot`

Request body:

```json
{
  "imageUrl": "https://example.com/image.png",
  "imageOcrText": ""
}
```

The endpoint will call:

```python
process_hotspot_image(
    request.imageUrl,
    HOTSPOT_MODEL,
    request.imageOcrText
)
```

This function is intended to run only for images already classified as `hotspot_image`.

---

### API 4 — Generate TOC

Future endpoint example:

`POST /generate-toc`

The request body will be the JSON returned from API 1.

The endpoint will call:

```python
generate_toc(
    extraction_result
)
```


## Full Colab Run — Process + Submit to Site

Run this cell after all previous cells. It performs extraction, image classification, conditional hotspot processing, TOC generation, then submits both blocks and TOC to the current website backend.

In [ ]:
# ==========================================================
# FULL COLAB TEST + SUBMIT TO CURRENT WEBSITE
# ==========================================================
# CHAPTER_ID is defined ONCE in the Configuration cell.
# Current target:
print("TARGET CHAPTER_ID:", CHAPTER_ID)

# 1) API 1 logic: extract objects
extraction_result = extract_objects_from_ebook(
    CHAPTER_ID,
    OBJECT_DETECTOR
)

# Safety check: never continue if a different chapter was returned.
if extraction_result.get("chapterId") != CHAPTER_ID:
    raise RuntimeError(
        "Chapter mismatch! "
        f"Expected {CHAPTER_ID} but extraction returned "
        f"{extraction_result.get('chapterId')}"
    )

print("Extraction chapter verified:", extraction_result["chapterId"])

# 2) API 2 + API 3 routing: classify images, then process hotspot images only
extraction_result = classify_all_extracted_images(
    extraction_result,
    IMAGE_CLASSIFIER,
    HOTSPOT_MODEL
)

# Safety check again before generating/submitting anything
if extraction_result.get("chapterId") != CHAPTER_ID:
    raise RuntimeError("Chapter ID changed before submission.")

# 3) API 4 logic: generate TOC
toc_result = generate_toc(
    extraction_result
)

if toc_result.get("chapterId") != CHAPTER_ID:
    raise RuntimeError(
        "TOC chapter mismatch! "
        f"Expected {CHAPTER_ID}, got {toc_result.get('chapterId')}"
    )

print("✅ TOC chapter verified:", toc_result["chapterId"])

# 4) Submit final blocks to current site
print("\n📤 Submitting blocks for chapter:", CHAPTER_ID)
site_block_responses = submit_extraction_result_to_site(
    extraction_result
)

# 5) Submit TOC to current site
print("\n📤 Submitting TOC for chapter:", CHAPTER_ID)
site_toc_response = submit_toc_to_site(
    toc_result
)

print(
    "\n✅ Processing finished and submission targeted chapter:",
    CHAPTER_ID
)


🎯 TARGET CHAPTER_ID: dbf7e22d74297ac6bd879949
🔄 Fetching pages for chapter: dbf7e22d74297ac6bd879949
✅ Found 20 pages
Response type: <class 'list'>
Response content: [{'_id': 'ced37fece7f0f56162d06a50', 'url': 'http://res.cloudinary.com/dd9turntq/image/upload/v1776775221/qfpveb6ylkiggiq13wtv.png', 'blocks': [], 'v_blocks': [], 'isNewPage': False}, {'_id': '6a732eec7cd3280004f0cefd', 'url': 'https://res.cloudinary.com/dapspj9n8/image/upload/v1785933797/question-bank/jpwanfyei6rdcova90vj.png', 'blocks': [], 'v_blocks': [], 'isNewPage': False}, {'_id': '7a9d8c44588fa48abebc451d', 'url': 'http://res.cloudinary.com/dd9turntq/image/upload/v1776775223/mcsttway30saqd2mz8jn.png', 'blocks': [], 'v_blocks': [], 'isNewPage': False}, {'_id': '5ece2754b0ba731b8e1b138e', 'url': 'http://res.cloudinary.com/dd9turntq/image/upload/v1776775226/wahrdevxlnnmaq9y7avj.png', 'blocks': [], 'v_blocks': [], 'isNewPage': False}, {'_id': '32719465a28f4b7769eb5a6b', 'url': 'http://res.cloudinary.com/dd9turntq/image/

Extracting page objects:  20%|██        | 4/20 [01:15<05:32, 20.75s/it]

https://scube-applications-media54cbabfc-u3d19945rbtv.s3.eu-west-1.amazonaws.com/new-interactive-objects/Page_004_image_0-e9c05da5-7550-4250-907e-3190e1416c5e.png


Extracting page objects:  25%|██▌       | 5/20 [01:43<05:50, 23.39s/it]

https://scube-applications-media54cbabfc-u3d19945rbtv.s3.eu-west-1.amazonaws.com/new-interactive-objects/Page_005_image_1-bcc5db79-8714-4137-a0a9-808b865c204b.png
    📤 Uploading merged image...


...

https://scube-applications-media54cbabfc-u3d19945rbtv.s3.eu-west-1.amazonaws.com/new-interactive-objects/Page_018_image_1-63622b21-6537-4af1-af19-2259a7ece485.png
    📤 Uploading merged image...


Extracting page objects:  95%|█████████▌| 19/20 [05:55<00:16, 16.39s/it]

https://scube-applications-media54cbabfc-u3d19945rbtv.s3.eu-west-1.amazonaws.com/new-interactive-objects/Page_018_merged_1_3-de48cde1-1bcc-4b18-9353-9adf4bafcb66.png
    ✅ Merged image uploaded: https://scube-applications-media54cbabfc-u3d19945rbtv.s3.eu-...
  ✅ Merged image with caption (below, 12px) on page 18


Extracting page objects: 100%|██████████| 20/20 [06:10<00:00, 18.54s/it]



✅ Extraction complete. Processed pages: 20
✅ Extraction chapter verified: dbf7e22d74297ac6bd879949
  🎯 Hotspot detected! Running hotspot processing...
  📍 Found 6 spots and 6 labels
  ✅ Hotspot matching: 6 pairs matched
  ✓ Spot #1: 'eevatrix'
  ✓ Spot #2: 'ribxosomcs'
  ✓ Spot #3: 'Inner membrane'
  ✓ Spot #4: 'Cristac'
  ✓ Spot #5: 'DNA'
  ✓ Spot #6: 'Outer membranc'
  🎯 Hotspot detected! Running hotspot processing...
  📍 Found 1 spots and 2 labels
  ✅ Hotspot matching: 1 pairs matched
  ✓ Spot #1: '70p'
  🎯 Hotspot detected! Running hotspot processing...
  📍 Found 1 spots and 4 labels
  ✅ Hotspot matching: 1 pairs matched
  ✓ Spot #1: 'Tfacilen'
  🎯 Hotspot detected! Running hotspot processing...
  📍 Found 0 spots and 0 labels
  ⚠️ No spots or labels found!
  🎯 Hotspot detected! Running hotspot processing...
  📍 Found 1 spots and 3 labels
  ✅ Hotspot matching: 1 pairs matched
  ✓ Spot #1: 'Light'
  🎯 Hotspot detected! Running hotspot processing...
  📍 Found 0 spots and 0 labels
  ⚠

## Debug Submission — Use This Instead of the Previous Submit Cell

This section does not assume that HTTP 200 means the data is visible.
It prints the complete server response, identifiers used for saving,
and then fetches the chapter again for verification.

In [ ]:
# ==========================================================
# DEBUG SUBMISSION LAYER
# ==========================================================

BLOCKS_ENDPOINT = "https://questions-api-navy.vercel.app/api/save-blocks"
TOC_ENDPOINT = "https://questions-api-navy.vercel.app/api/composite-objects"


def submit_blocks_to_site_debug(blocks):
    """
    Submit one page's blocks and print the REAL server response.
    """

    if not blocks:
        print(" No blocks to submit.")
        return None

    page_ids = sorted({
        str(block.get("pageId"))
        for block in blocks
        if block.get("pageId")
    })

    payload = {"blocks": blocks}

    print("\n" + "=" * 70)
    print("SUBMITTING BLOCKS")
    print("Endpoint:", BLOCKS_ENDPOINT)
    print("Blocks count:", len(blocks))
    print("pageId(s):", page_ids)
    print("First block:")
    print(json.dumps(blocks[0], ensure_ascii=False, indent=2))

    try:
        response = requests.post(
            BLOCKS_ENDPOINT,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=120
        )

        print("HTTP status:", response.status_code)
        print("Raw response:")
        print(response.text[:5000])

        response.raise_for_status()

        try:
            result = response.json()
        except Exception:
            result = {"raw_text": response.text}

        # Do not treat HTTP 200 alone as proof of persistence.
        if isinstance(result, dict):
            if result.get("success") is False:
                print("Server returned success=false")
            elif result.get("error"):
                print("Server returned an error field:", result.get("error"))
            else:
                print("HTTP request accepted by server.")
        else:
            print("HTTP request accepted by server.")

        return result

    except Exception as e:
        print("Submission exception:", repr(e))
        return None


def submit_toc_to_site_debug(toc_result):
    """
    Submit TOC and print the REAL server response.
    """

    if not toc_result:
        print("⚠️ No TOC to submit.")
        return None

    print("\n" + "=" * 70)
    print("📤 SUBMITTING TOC")
    print("Endpoint:", TOC_ENDPOINT)
    print("chapterId:", toc_result.get("chapterId"))
    print("pageId:", toc_result.get("pageId"))
    print("TOC entries:", len(toc_result.get("blocks", [])))
    print("TOC payload:")
    print(json.dumps(toc_result, ensure_ascii=False, indent=2)[:5000])

    try:
        response = requests.post(
            TOC_ENDPOINT,
            json=toc_result,
            headers={"Content-Type": "application/json"},
            timeout=300
        )

        print("HTTP status:", response.status_code)
        print("Raw response:")
        print(response.text[:5000])

        response.raise_for_status()

        try:
            result = response.json()
        except Exception:
            result = {"raw_text": response.text}

        if isinstance(result, dict):
            if result.get("success") is False:
                print("Server returned success=false")
            elif result.get("error"):
                print("Server returned an error field:", result.get("error"))
            else:
                print("✅ HTTP request accepted by server.")
        else:
            print("✅ HTTP request accepted by server.")

        return result

    except Exception as e:
        print("TOC submission exception:", repr(e))
        return None


def submit_extraction_result_to_site_debug(extraction_result):
    """
    Convert every page to site schema and submit it with diagnostics.
    """

    if not extraction_result:
        print("No extraction result available.")
        return []

    print("\nChapter being submitted:", extraction_result.get("chapterId"))
    print("Pages in extraction result:", len(extraction_result.get("pages", [])))

    responses = []

    for page in extraction_result.get("pages", []):
        page_id = page.get("pageId")
        objects = page.get("objects", [])

        print(
            f"\n Page {page.get('pageIndex', 0) + 1} | "
            f"pageId={page_id} | raw objects={len(objects)}"
        )

        if not page_id:
            print("Skipped: pageId is empty.")
            continue

        if not objects:
            print("Skipped: no objects.")
            continue

        blocks = convert_to_api_format(
            objects,
            page_id,
            page.get("pageWidth", 0),
            page.get("pageHeight", 0)
        )

        result = submit_blocks_to_site_debug(blocks)

        responses.append({
            "pageId": page_id,
            "response": result
        })

    return responses


def inspect_chapter_from_api(chapter_id):
    """
    Fetch the chapter again after submission and show exactly
    what the page API currently returns.
    """
    print("\n" + "=" * 70)
    print("🔎 VERIFYING CHAPTER FROM API")
    print("chapterId:", chapter_id)

    try:
        response = requests.get(
            PAGES_API_BASE,
            params={"chapterId": chapter_id},
            timeout=60
        )

        print("GET status:", response.status_code)
        response.raise_for_status()

        pages = response.json()

        print("Pages returned:", len(pages))

        if pages:
            print("First page keys:", list(pages[0].keys()))
            print("First page _id/id/pageId:",
                  pages[0].get("_id"),
                  pages[0].get("id"),
                  pages[0].get("pageId"))

            # Show persistence-related fields if the API exposes them.
            for key in ["blocks", "objects", "content", "compositeObjects"]:
                if key in pages[0]:
                    value = pages[0].get(key)
                    try:
                        size = len(value) if value is not None else 0
                    except Exception:
                        size = "n/a"
                    print(f"First page field '{key}' size:", size)

        return pages

    except Exception as e:
        print("Verification GET failed:", repr(e))
        return None

## Run Debug Submission

In [ ]:
# ==========================================================
# DEBUG SUBMISSION
# ==========================================================

print("🎯 Configured CHAPTER_ID:", CHAPTER_ID)
print("📦 extraction_result chapterId:", extraction_result.get("chapterId"))
print("📚 toc_result chapterId:", toc_result.get("chapterId"))

assert extraction_result.get("chapterId") == CHAPTER_ID, (
    f"Extraction mismatch: expected {CHAPTER_ID}, "
    f"got {extraction_result.get('chapterId')}"
)

assert toc_result.get("chapterId") == CHAPTER_ID, (
    f"TOC mismatch: expected {CHAPTER_ID}, "
    f"got {toc_result.get('chapterId')}"
)

# 1) Submit blocks with full diagnostics
debug_block_responses = submit_extraction_result_to_site_debug(
    extraction_result
)

# 2) Submit TOC with full diagnostics
debug_toc_response = submit_toc_to_site_debug(
    toc_result
)

# 3) Verify THE SAME chapter from backend
verified_pages = inspect_chapter_from_api(
    CHAPTER_ID
)


🎯 Configured CHAPTER_ID: dbf7e22d74297ac6bd879949
📦 extraction_result chapterId: dbf7e22d74297ac6bd879949
📚 toc_result chapterId: dbf7e22d74297ac6bd879949

Chapter being submitted: dbf7e22d74297ac6bd879949
Pages in extraction result: 20

📦 Page 1 | pageId=ced37fece7f0f56162d06a50 | raw objects=3

📤 SUBMITTING BLOCKS
Endpoint: https://questions-api-navy.vercel.app/api/save-blocks
Blocks count: 1
pageId(s): ['ced37fece7f0f56162d06a50']
First block:
{
  "pageId": "ced37fece7f0f56162d06a50",
  "status": "new",
  "coordinates": {
    "x": 16.26,
    "y": 51.47,
    "width": 66.26,
    "height": 23.63,
    "unit": "percentage"
  },
  "contentType": "Paragraph",
  "contentValue": "Identify the concept of cellular respiration  Understand the steps of glycolysis and its importance Identify the steps of aerobic respiration and where it takes place  Distinguish between aerobic and anaerobic respiration  State the importance of cellular respiration  Link between photosynthesis and respiration in p

## Verify Illustrative Object Payload\n\nRun this before submission. Image blocks should show `objectType` and `itemType` as `Illustrative object`.\n

In [ ]:

for page in extraction_result.get("pages", []):
    test_blocks = convert_to_api_format(
        page["objects"],
        page["pageId"],
        page["pageWidth"],
        page["pageHeight"]
    )

    image_blocks = [
        b for b in test_blocks
        if b.get("contentType") == "image"
    ]

    if image_blocks:
        print(json.dumps(image_blocks[0], ensure_ascii=False, indent=2))
        break
